# 🔍 Grouped-Query Attention (GQA): Complete Hands-On Lab

Welcome! In this notebook, we explore <b>Grouped-Query Attention (GQA)</b>—the attention architecture used in <b>Llama 3, Mistral, and Mirendil-Base</b> that cuts Key-Value memory by up to <b>87.5%</b> while preserving model accuracy.

---

## 1. The Core Metaphor: The Office Library

Imagine an AI research office with <b>12 Analysts</b> (Query heads), each looking for different code patterns.<br>
To do their jobs, they must look up past tokens stored in reference cabinets (<b>Key / Value Cache</b>).

<pre style='font-size: 14px; background: #0d1117; color: #c9d1d9; padding: 16px; border-radius: 8px; border: 1px solid #30363d;'>
1. MHA (Old Way - Multi-Head Attention):
   Every single Analyst keeps their own private copy of the cabinet!
   12 Analysts = 12 Filing Cabinets (100% Memory Cost - GPU Runs Out of RAM!)

2. MQA (The Extreme Cheap Way - Multi-Query Attention):
   All 12 Analysts share only 1 single filing cabinet.
   (Saves 92% memory, but model intelligence drops noticeably!)

3. GQA (The Modern Winner - Grouped-Query Attention):
   Group the 12 analysts into 4 pods of 3. Each pod shares 1 cabinet!
   12 Analysts &rarr; 4 Cabinets (66.7% Memory Saved with virtually 0 quality loss!)
</pre>

---
## 2. If Multiple Query Heads Share 1 Key/Value, Why Not Just Use 1 Head?

<b>The Metaphor: The Same Reference Book, 3 Different Questions</b>
<br>
Think of the shared Key & Value as a single reference document: a Python function definition.<br>
Three different Query heads read the <b>same book</b>, but look for <b>different things</b>:
<ul>
  <li><b>Query Q0 asks:</b> <i>"What is the function's name?"</i> &rarr; Focuses on <code>calculate_loss</code></li>
  <li><b>Query Q1 asks:</b> <i>"What are the arguments?"</i> &rarr; Focuses on <code>pred, target</code></li>
  <li><b>Query Q2 asks:</b> <i>"What is the return type?"</i> &rarr; Focuses on <code>Tensor</code></li>
</ul>

<pre style='font-size: 13px; background: #0d1117; color: #58a6ff; padding: 16px; border-radius: 8px; border: 1px solid #30363d;'>
                  Single Shared Reference Token:
                  Key:   [ "Function Definition" ]
                  Value: [ "def calculate_loss(pred, target) -> Tensor:" ]
                                     │
         ┌───────────────────────────┼───────────────────────────┐
         ▼                           ▼                           ▼
    Analyst Q0                  Analyst Q1                  Analyst Q2
  "Look for Name"           "Look for Inputs"           "Look for Output"
         │                           │                           │
  Dot Product with K          Dot Product with K          Dot Product with K
  High score on:              High score on:              High score on:
  ['calculate_loss']         ['pred', 'target']               ['Tensor']
         │                           │                           │
         ▼                           ▼                           ▼
   Head 0 Output               Head 1 Output               Head 2 Output
  (Knows the Name)            (Knows the Inputs)          (Knows the Type)
</pre>

---
## 3. Exercise 1: Same Key, Different Query Scores on Paper 📝

Suppose our shared Key vector is:
<pre style='font-size: 15px; background: #161b22; color: #bc8cff; padding: 12px; border-radius: 8px;'>
Key K = [ 1.0,  0.0 ]
</pre>

And our two Query heads ask different questions:
<pre style='font-size: 15px; background: #161b22; color: #58a6ff; padding: 12px; border-radius: 8px;'>
Query Q0 = [ 1.0,  0.0 ]   (Looking for feature A)
Query Q1 = [ 0.0,  1.0 ]   (Looking for feature B)
</pre>

### Dot Product Attention Scores on Paper:
1. <b>Head 0:</b> Q0 · K = (1.0 · 1.0) + (0.0 · 0.0) = <b>1.0</b> &rarr; (100% Match! Attends heavily)
2. <b>Head 1:</b> Q1 · K = (0.0 · 1.0) + (1.0 · 0.0) = <b>0.0</b> &rarr; (0% Match! Ignores it completely)

<br>
<b>Conclusion:</b> Sharing the Key saves 50% memory, but both heads maintain complete freedom to pay attention to different parts of the sentence!

In [ ]:
import torch

shared_key = torch.tensor([1.0, 0.0])
q0 = torch.tensor([1.0, 0.0])  # Head 0
q1 = torch.tensor([0.0, 1.0])  # Head 1

score_0 = (q0 * shared_key).sum().item()
score_1 = (q1 * shared_key).sum().item()

print(f"Head 0 Attention Score: {score_0:.1f} (Full Attention)")
print(f"Head 1 Attention Score: {score_1:.1f} (Zero Attention)")

---
## 4. What is `repeat_interleave`? (The Photocopying Analogy) 📝

Suppose you have 2 books: <b>[ Math, Physics ]</b>.<br>
You have 6 students in 2 pods: Pod 1 (Students 1, 2, 3) needs Math; Pod 2 (Students 4, 5, 6) needs Physics.<br><br>

Look at the critical difference between standard <b>repeat</b> vs <b>repeat_interleave</b>:

<pre style='font-size: 14px; background: #0d1117; color: #c9d1d9; padding: 14px; border-radius: 8px; border: 1px solid #30363d;'>
Initial List: [ A, B ]

❌ Standard repeat(3) tiles the entire list:
   [ A, B,   A, B,   A, B ]   &lt;-- WRONG! Alternates between A and B

✅ repeat_interleave(3) duplicates each element next to itself:
   [ A, A, A,   B, B, B ]     &lt;-- CORRECT! A is for Pod 1, B is for Pod 2
</pre>

### In GQA with 4 Key Heads and 12 Query Heads:
<pre style='font-size: 14px; background: #161b22; color: #bc8cff; padding: 12px; border-radius: 8px;'>
Original 4 KV heads:  [ KV0,  KV1,  KV2,  KV3 ]

repeat_interleave(3) produces:
[ KV0, KV0, KV0,   KV1, KV1, KV1,   KV2, KV2, KV2,   KV3, KV3, KV3 ]
  └─ Pod 1 Qs ─┘     └─ Pod 2 Qs ─┘     └─ Pod 3 Qs ─┘     └─ Pod 4 Qs ─┘
</pre>

In [ ]:
kv_heads = torch.tensor([[10], [20], [30], [40]])  # 4 heads

# repeat_interleave duplicates each element consecutively
expanded_kv = kv_heads.repeat_interleave(3, dim=0)

print(f"Original 4 KV heads:\n{kv_heads.squeeze().tolist()}")
print(f"\nExpanded 12 KV heads via repeat_interleave(3):\n{expanded_kv.squeeze().tolist()}")
assert len(expanded_kv) == 12
print("\nSUCCESS: Interleave repetition verified!")

---
## 5. Verify in PyTorch with Real Project GQA Code

Now let's trace a forward pass through our project's actual `GroupedQueryAttention` class from `src/model/attention.py`:

In [ ]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import torch
from src.model.attention import GroupedQueryAttention
from src.model.rope import precompute_freqs_cis

# Setup GQA with 12 Query heads and 4 KV heads
dim = 768
n_heads = 12
n_kv_heads = 4
seq_len = 8

gqa = GroupedQueryAttention(dim=dim, n_heads=n_heads, n_kv_heads=n_kv_heads)
cos, sin = precompute_freqs_cis(dim // n_heads, max_seq_len=seq_len)

print(f"Q projection matrix shape:  {list(gqa.q_proj.weight.shape)} (12 heads)")
print(f"K projection matrix shape:  {list(gqa.k_proj.weight.shape)}  (4 heads - 66.7% smaller!)")
print(f"V projection matrix shape:  {list(gqa.v_proj.weight.shape)}  (4 heads - 66.7% smaller!)")

# Forward pass on dummy batch
x = torch.randn(2, seq_len, dim)
out = gqa(x, cos, sin)

print(f"\nInput Tensor Shape:        {list(x.shape)}")
print(f"Attention Output Shape:    {list(out.shape)}")
assert out.shape == x.shape
print("\nSUCCESS: GQA Forward pass verified completely!")